# Notebook v3 release — Full data / 12 methods

เลือก GPU บน Colab แล้วรันตามลำดับ อัปโหลด `Notebook_3_release_colab.zip` (โค้ดเท่านั้น)
ไป `MyDrive/nih -xray/` และอัปโหลด dataset ZIP 13 ส่วนพร้อม index ไป `MyDrive/nih -xray/data/v3_selected_zip/`
Setup คัดลอก ZIP จาก Drive แล้วแตกภาพต้นฉบับลง `/content/cxr_v3_originals` ก่อนสร้าง cache; ดู `ZIP_DATA_GUIDE.md`

Train 46,108 / Validation 11,719 / Test 12,081; XAI123ภาพ ทุก method ใช้ manifests เดียวกัน
ค่าเริ่มต้น3seeds/20epochs รวม36CNN+3DAE; ไม่ใช่ notebook ที่รันเต็มได้ในไม่กี่นาที
Results/checkpoints ลง Drive; input cache อยู่ VM และอาจต้องสร้างใหม่เมื่อเปลี่ยน runtime
ทำซ้ำ cell เพื่อ resume; ถ้า config/code/dependency versions เปลี่ยน ใช้ output directory ใหม่


In [ ]:
from pathlib import Path
import importlib, importlib.util, sys, zipfile, shutil, subprocess

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = Path("/content/drive/MyDrive/nih -xray/data/3")
    RELEASE_ROOT = Path("/content/drive/MyDrive/nih -xray/Notebook_3_release")
    CODE_ZIP = Path("/content/drive/MyDrive/nih -xray/Notebook_3_release_colab.zip")
    if CODE_ZIP.is_file():
        ROOT = Path("/content/Notebook_3_release")
        with zipfile.ZipFile(CODE_ZIP) as archive:
            for member in archive.infolist():
                relative = Path(member.filename)
                if relative.is_absolute() or ".." in relative.parts or not relative.parts or relative.parts[0] != "Notebook_3_release":
                    raise ValueError("Invalid source package path")
                if member.is_dir():
                    continue
                target = Path("/content").joinpath(*relative.parts).resolve()
                if not target.is_relative_to(ROOT.resolve()):
                    raise ValueError("Invalid source package destination")
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, target.open("wb") as destination:
                    shutil.copyfileobj(source, destination)
    elif (RELEASE_ROOT / "pipeline.py").is_file():
        ROOT = RELEASE_ROOT
    else:
        raise FileNotFoundError("Upload Notebook_3_release_colab.zip to MyDrive/nih -xray, or upload the source folder as Notebook_3_release")
    import runpy
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")])
    runpy.run_path(str(ROOT / "tools/check_environment.py"))["check_numpy"]()
    ZIP_ROOT = Path("/content/drive/MyDrive/nih -xray/data/v3_selected_zip")
    stage_dataset = runpy.run_path(str(ROOT / "tools/stage_dataset.py"))["stage_dataset"]
    DATA_ROOT = stage_dataset(ROOT, ZIP_ROOT, Path("/content/cxr_v3_originals"))

    OUTPUT_ROOT = RELEASE_ROOT / "runs/release_full"
    CACHE_ROOT = Path("/content/cxr_v3_cache")
else:
    candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "Notebook_3 release"]
    ROOT = next((p for p in candidates if (p / "configs/release_full.json").is_file()), None)
    if ROOT is None:
        raise FileNotFoundError("Open from Notebook_3 release or its repository directory")
    DATA_ROOT = None  # locate local data/versions/3 without changing the frozen manifests
    OUTPUT_ROOT = ROOT / "runs/release_full"
    CACHE_ROOT = ROOT / ".runtime/cache"

sys.path.insert(0, str(ROOT))
for name in tuple(sys.modules):
    if name == "shared" or name.startswith("shared.") or name in ("pipeline", "check_overfit", "export_results_colab"):
        del sys.modules[name]
importlib.invalidate_caches()
import torch
from pipeline import open_run, run_stages
SEEDS = [42, 43, 44]
SMOKE = False  # True = isolated tiny validation run with random initialization; not research scores
CONDITIONS = None  # None = all 12 controlled conditions
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Data:", DATA_ROOT, "| Results:", OUTPUT_ROOT, "| Device:", DEVICE)


## 1. ตรวจข้อมูลและเตรียม working cache บน VM

In [ ]:
import threading
import time
from contextlib import contextmanager

@contextmanager
def preparation_progress(seed, interval=10):
    """Notebook-only observer; preserve all source files, pixels and run fingerprints."""
    import shared.data as data_module
    from shared.runtime_cache import RuntimeCache
    started = time.monotonic()
    status = {"phase": "opening run / checking configuration", "done": 0, "total": 0,
              "image": None, "reading_since": None, "reads": 0, "first_read": None}
    stop = threading.Event()
    original_manifests = data_module.prepare_manifests
    original_verify = data_module.verify_manifests
    original_prepare = RuntimeCache.prepare
    original_read = RuntimeCache._read

    def emit(message):
        print(f"[prepare seed={seed}] {message}", flush=True)

    def heartbeat():
        while not stop.wait(interval):
            elapsed = time.monotonic() - started
            message = f"{status['phase']} | elapsed {elapsed:.0f}s"
            if status["total"]:
                message += f" | {status['done']:,}/{status['total']:,} ({100*status['done']/status['total']:.1f}%)"
            if status["image"]:
                message += f" | image {status['image']}"
                if status["reading_since"] is not None:
                    message += f" | current read {time.monotonic()-status['reading_since']:.0f}s"
            if status["first_read"] is not None:
                rate = status["reads"] / max(time.monotonic()-status["first_read"], .001)
                message += f" | newly read {status['reads']:,} | {rate:.2f} images/s"
            emit(message)

    def manifests(exp):
        status["phase"] = "checking source checksums and loading frozen manifests"
        emit(status["phase"] + f" | data={exp.data_root}")
        result = original_manifests(exp)
        emit("manifests verified: " + str({name: len(frame) for name, frame in result.items()}))
        return result

    def verify(exp, frames):
        status["phase"] = "validating labels, patient partitions and bounding boxes"
        emit(status["phase"])
        result = original_verify(exp, frames)
        status["phase"] = "saving/verifying run manifest records"
        return result

    def prepare(cache, frames):
        status.update(phase="checking runtime cache and Drive folder layout", done=0,
                      total=len(set().union(*(set(frame.image) for frame in frames.values()))),
                      image=None, reading_since=None, reads=0, first_read=None)
        emit(f"source cache: {status['total']:,} unique images | cache={cache.root}")
        result = original_prepare(cache, frames)
        status.update(phase="cache verified and ready", done=status["total"], image=None, reading_since=None)
        emit(f"cache ready: {status['total']:,} images (existing verified shards are reused)")
        return result

    def read(cache, row):
        status.update(phase="reading source image / resizing / caching", image=row["image"], reading_since=time.monotonic())
        if status["first_read"] is None:
            status["first_read"] = time.monotonic()
            emit(f"first image read: {row['image']}; heartbeat every {interval}s")
        result = original_read(cache, row)
        status["reads"] += 1
        status["done"] = max(status["done"], cache.indices[row["image"]]+1)
        status["reading_since"] = None
        return result

    data_module.prepare_manifests = manifests
    data_module.verify_manifests = verify
    RuntimeCache.prepare = prepare
    RuntimeCache._read = read
    monitor = threading.Thread(target=heartbeat, daemon=True)
    emit("START: verifying data and preparing cache; model training has not started")
    monitor.start()
    try:
        yield
    finally:
        stop.set()
        monitor.join(timeout=1)
        data_module.prepare_manifests = original_manifests
        data_module.verify_manifests = original_verify
        RuntimeCache.prepare = original_prepare
        RuntimeCache._read = original_read
        emit(f"preparation cell ended after {time.monotonic()-started:.1f}s")

runs = {}
for seed in SEEDS:
    with preparation_progress(seed):
        exp, manifests = open_run(DATA_ROOT, OUTPUT_ROOT, CACHE_ROOT, seed=seed, smoke=SMOKE)
    runs[seed] = (exp, manifests)
    print(seed, {name: len(frame) for name, frame in manifests.items()}, exp.output_root, flush=True)


## 2. ฝึก DAE จาก train เท่านั้น

### GPU batch tuning — เพิ่ม DAE batch จากผลวัด VRAM/ความเร็ว
ใช้ VRAM budget90%; เปลี่ยน batch แล้วเริ่ม output ใหม่ เก็บผลเดิมไว้ ตั้ง USE_GPU_DAE_BATCH=False เพื่อใช้แผนเดิม

In [ ]:
USE_GPU_DAE_BATCH = not SMOKE
if USE_GPU_DAE_BATCH:
    # Run AFTER setup + preparation, BEFORE DAE. Stop the existing DAE cell first.
    # Preserve old outputs. A changed batch starts a separate experiment from epoch 1.
    import runpy
    from pathlib import Path
    gpu_batch_tools = runpy.run_path(str(Path(ROOT) / "tools/dae_gpu_batch.py"))
    gpu_batch_probe = gpu_batch_tools["benchmark_dae"](next(iter(runs.values()))[0].config,
                                                     fraction=.90, steps=3)
    runs_before_gpu_batch = runs
    runs = gpu_batch_tools["open_batch_runs"](ROOT, runs_before_gpu_batch, gpu_batch_probe)
    OUTPUT_ROOT = next(iter(runs.values()))[0].base_output_root
    print("Use these runs for DAE, CNN, XAI and reports. New outputs:", OUTPUT_ROOT, flush=True)


In [ ]:
import threading
import time
import inspect
from contextlib import contextmanager
from functools import wraps

@contextmanager
def training_progress(seed, stage, interval=10):
    """Observe original training; do not change model, optimizer, batches or checkpoints."""
    started = time.monotonic()
    status = {"phase": "importing training modules", "epoch": 0, "batch": 0,
              "total": 0, "images": 0, "since": started, "condition": stage}
    stop = threading.Event()
    patches = []
    def emit(message):
        print(f"[seed={seed} {stage}] {message}", flush=True)
    def heartbeat():
        while not stop.wait(interval):
            message = f"{status['condition']} | {status['phase']} | elapsed {time.monotonic()-started:.0f}s"
            if status['total']:
                seconds = max(time.monotonic()-status['since'], .001)
                rate = status['images']/seconds
                remaining = (status['total']-status['batch'])*seconds/max(status['batch'], 1)
                message += f" | epoch {status['epoch']} | batch {status['batch']:,}/{status['total']:,} | {status['images']:,} images | {rate:.1f} images/s"
                if status['batch']:
                    message += f" | pass ETA ~{remaining/60:.1f} min"
            emit(message)
    def patch(module, name, replacement):
        patches.append((module, name, getattr(module, name)))
        setattr(module, name, replacement)
    class ObservedBatches:
        def __init__(self, original):
            self.original = original
        def __len__(self):
            return len(self.original)
        def __getattr__(self, name):
            return getattr(self.original, name)
        def __iter__(self):
            for batch in self.original:
                yield batch
                # Advance only after the caller finished processing this batch.
                status['batch'] += 1
                status['images'] += len(batch[0])
    monitor = threading.Thread(target=heartbeat, daemon=True)
    emit("START: training observer active; heartbeat every 10s")
    monitor.start()
    try:
        import shared.training as training
        from shared.preprocessing import ReleasePreprocessor
        original_prepare = ReleasePreprocessor.prepare
        def prepare(preprocessor, frames):
            status.update(phase='checking/building preprocessing cache on VM', total=0)
            emit(status['phase'])
            result = original_prepare(preprocessor, frames)
            status['phase'] = 'preprocessing ready / preparing model'
            emit(status['phase'])
            return result
        patch(ReleasePreprocessor, 'prepare', prepare)
        def observe_epoch(name):
            original = getattr(training, name)
            @wraps(original)
            def wrapped(model, batches, *args, **kwargs):
                if name == '_reconstruction_epoch':
                    optimizer = kwargs.get('optimizer', args[2] if len(args)>2 else None)
                    training_pass = optimizer is not None
                    phase = 'DAE training' if training_pass else 'DAE validation / class validation'
                else:
                    training_pass = name == 'optimization_epoch'
                    phase = {'optimization_epoch': 'CNN training', 'balanced_loss_epoch': 'CNN balanced-loss evaluation', 'predict': 'classifier predictions'}[name]
                if training_pass:
                    status['epoch'] += 1
                status.update(phase=phase, batch=0, total=len(batches), images=0, since=time.monotonic())
                emit(f"{status['condition']} | {phase} | epoch {status['epoch']} | {len(batches):,} batches")
                result = original(model, ObservedBatches(batches), *args, **kwargs)
                emit(f"{phase} finished | {status['images']:,} images | {time.monotonic()-status['since']:.1f}s")
                status['total'] = 0
                return result
            return wrapped
        for name in ('_reconstruction_epoch', 'optimization_epoch', 'balanced_loss_epoch', 'predict'):
            patch(training, name, observe_epoch(name))
        def observe_operation(name, label):
            original = getattr(training, name)
            @wraps(original)
            def wrapped(*args, **kwargs):
                status.update(phase=label, total=0)
                emit(label)
                result = original(*args, **kwargs)
                if name == '_restore_training':
                    status['epoch'] = int(result[0])
                    emit(f"resumed completed epoch {status['epoch']}")
                    status['phase'] = 'checkpoint restored / preparing epoch'
                elif name == '_completed':
                    emit('completed training verified; reuse' if result is not None else 'training not complete; initialize/resume')
                    status['phase'] = 'verified completed training' if result is not None else 'initializing/resuming training'
                elif name == '_resume_checkpoint':
                    status['phase'] = 'checkpoint found / restoring state' if result is not None else 'no checkpoint / preparing first epoch'
                return result
            return wrapped
        for name, label in (('_completed','checking completion and Drive hashes'),
                            ('_resume_checkpoint','checking/loading resumable checkpoint from Drive'),
                            ('_restore_training','restoring model / optimizer / RNG')):
            patch(training, name, observe_operation(name, label))
        original_save = training.save_torch
        def save(path, state):
            status.update(phase=f'saving checkpoint to Drive: {path}', total=0)
            emit(status['phase'])
            result = original_save(path, state)
            emit(f"checkpoint saved | epoch {state.get('epoch', '?')}")
            status['phase'] = 'checkpoint saved / finalizing epoch'
            return result
        patch(training, 'save_torch', save)
        def observe_training(name):
            original = getattr(training, name)
            @wraps(original)
            def wrapped(exp, manifests, *args, **kwargs):
                condition = 'DAE' if name == 'train_dae' else (args[0] if args else kwargs.get('condition_id', 'CNN'))
                status.update(condition=condition, epoch=0, total=0, phase='checking run provenance / initializing model')
                emit(f"{condition} START | device={kwargs.get('device', 'cuda')} | train={len(manifests['train']):,}")
                if name == 'train_condition':
                    factory = kwargs.get('model_factory', inspect.signature(original).parameters['model_factory'].default)
                    def model_factory(*a, **k):
                        status.update(phase='initializing ResNet50 / loading ImageNet weights or resume model', total=0)
                        emit(status['phase'])
                        result = factory(*a, **k)
                        status['phase'] = 'model ready / preparing optimizer'
                        return result
                    kwargs['model_factory'] = model_factory
                result = original(exp, manifests, *args, **kwargs)
                status.update(phase=f'{condition} training function returned', total=0)
                emit(status['phase'])
                return result
            return wrapped
        for name in ('train_dae', 'train_condition'):
            patch(training, name, observe_training(name))
        yield
    finally:
        stop.set()
        monitor.join(timeout=1)
        for module, name, original in reversed(patches):
            setattr(module, name, original)
        emit(f"cell ended after {time.monotonic()-started:.1f}s; training hooks restored")

TRAIN_STAGE = "dae"
for seed, (exp, manifests) in runs.items():
    with training_progress(seed, TRAIN_STAGE):
        run_stages(exp, manifests, stages=[TRAIN_STAGE], condition_ids=CONDITIONS, device=DEVICE)


## 3. ฝึก classifier ทั้ง12methods

In [ ]:
import threading
import time
import inspect
from contextlib import contextmanager
from functools import wraps

@contextmanager
def training_progress(seed, stage, interval=10):
    """Observe original training; do not change model, optimizer, batches or checkpoints."""
    started = time.monotonic()
    status = {"phase": "importing training modules", "epoch": 0, "batch": 0,
              "total": 0, "images": 0, "since": started, "condition": stage}
    stop = threading.Event()
    patches = []
    def emit(message):
        print(f"[seed={seed} {stage}] {message}", flush=True)
    def heartbeat():
        while not stop.wait(interval):
            message = f"{status['condition']} | {status['phase']} | elapsed {time.monotonic()-started:.0f}s"
            if status['total']:
                seconds = max(time.monotonic()-status['since'], .001)
                rate = status['images']/seconds
                remaining = (status['total']-status['batch'])*seconds/max(status['batch'], 1)
                message += f" | epoch {status['epoch']} | batch {status['batch']:,}/{status['total']:,} | {status['images']:,} images | {rate:.1f} images/s"
                if status['batch']:
                    message += f" | pass ETA ~{remaining/60:.1f} min"
            emit(message)
    def patch(module, name, replacement):
        patches.append((module, name, getattr(module, name)))
        setattr(module, name, replacement)
    class ObservedBatches:
        def __init__(self, original):
            self.original = original
        def __len__(self):
            return len(self.original)
        def __getattr__(self, name):
            return getattr(self.original, name)
        def __iter__(self):
            for batch in self.original:
                yield batch
                # Advance only after the caller finished processing this batch.
                status['batch'] += 1
                status['images'] += len(batch[0])
    monitor = threading.Thread(target=heartbeat, daemon=True)
    emit("START: training observer active; heartbeat every 10s")
    monitor.start()
    try:
        import shared.training as training
        from shared.preprocessing import ReleasePreprocessor
        original_prepare = ReleasePreprocessor.prepare
        def prepare(preprocessor, frames):
            status.update(phase='checking/building preprocessing cache on VM', total=0)
            emit(status['phase'])
            result = original_prepare(preprocessor, frames)
            status['phase'] = 'preprocessing ready / preparing model'
            emit(status['phase'])
            return result
        patch(ReleasePreprocessor, 'prepare', prepare)
        def observe_epoch(name):
            original = getattr(training, name)
            @wraps(original)
            def wrapped(model, batches, *args, **kwargs):
                if name == '_reconstruction_epoch':
                    optimizer = kwargs.get('optimizer', args[2] if len(args)>2 else None)
                    training_pass = optimizer is not None
                    phase = 'DAE training' if training_pass else 'DAE validation / class validation'
                else:
                    training_pass = name == 'optimization_epoch'
                    phase = {'optimization_epoch': 'CNN training', 'balanced_loss_epoch': 'CNN balanced-loss evaluation', 'predict': 'classifier predictions'}[name]
                if training_pass:
                    status['epoch'] += 1
                status.update(phase=phase, batch=0, total=len(batches), images=0, since=time.monotonic())
                emit(f"{status['condition']} | {phase} | epoch {status['epoch']} | {len(batches):,} batches")
                result = original(model, ObservedBatches(batches), *args, **kwargs)
                emit(f"{phase} finished | {status['images']:,} images | {time.monotonic()-status['since']:.1f}s")
                status['total'] = 0
                return result
            return wrapped
        for name in ('_reconstruction_epoch', 'optimization_epoch', 'balanced_loss_epoch', 'predict'):
            patch(training, name, observe_epoch(name))
        def observe_operation(name, label):
            original = getattr(training, name)
            @wraps(original)
            def wrapped(*args, **kwargs):
                status.update(phase=label, total=0)
                emit(label)
                result = original(*args, **kwargs)
                if name == '_restore_training':
                    status['epoch'] = int(result[0])
                    emit(f"resumed completed epoch {status['epoch']}")
                    status['phase'] = 'checkpoint restored / preparing epoch'
                elif name == '_completed':
                    emit('completed training verified; reuse' if result is not None else 'training not complete; initialize/resume')
                    status['phase'] = 'verified completed training' if result is not None else 'initializing/resuming training'
                elif name == '_resume_checkpoint':
                    status['phase'] = 'checkpoint found / restoring state' if result is not None else 'no checkpoint / preparing first epoch'
                return result
            return wrapped
        for name, label in (('_completed','checking completion and Drive hashes'),
                            ('_resume_checkpoint','checking/loading resumable checkpoint from Drive'),
                            ('_restore_training','restoring model / optimizer / RNG')):
            patch(training, name, observe_operation(name, label))
        original_save = training.save_torch
        def save(path, state):
            status.update(phase=f'saving checkpoint to Drive: {path}', total=0)
            emit(status['phase'])
            result = original_save(path, state)
            emit(f"checkpoint saved | epoch {state.get('epoch', '?')}")
            status['phase'] = 'checkpoint saved / finalizing epoch'
            return result
        patch(training, 'save_torch', save)
        def observe_training(name):
            original = getattr(training, name)
            @wraps(original)
            def wrapped(exp, manifests, *args, **kwargs):
                condition = 'DAE' if name == 'train_dae' else (args[0] if args else kwargs.get('condition_id', 'CNN'))
                status.update(condition=condition, epoch=0, total=0, phase='checking run provenance / initializing model')
                emit(f"{condition} START | device={kwargs.get('device', 'cuda')} | train={len(manifests['train']):,}")
                if name == 'train_condition':
                    factory = kwargs.get('model_factory', inspect.signature(original).parameters['model_factory'].default)
                    def model_factory(*a, **k):
                        status.update(phase='initializing ResNet50 / loading ImageNet weights or resume model', total=0)
                        emit(status['phase'])
                        result = factory(*a, **k)
                        status['phase'] = 'model ready / preparing optimizer'
                        return result
                    kwargs['model_factory'] = model_factory
                result = original(exp, manifests, *args, **kwargs)
                status.update(phase=f'{condition} training function returned', total=0)
                emit(status['phase'])
                return result
            return wrapped
        for name in ('train_dae', 'train_condition'):
            patch(training, name, observe_training(name))
        yield
    finally:
        stop.set()
        monitor.join(timeout=1)
        for module, name, original in reversed(patches):
            setattr(module, name, original)
        emit(f"cell ended after {time.monotonic()-started:.1f}s; training hooks restored")

TRAIN_STAGE = "cnn"
for seed, (exp, manifests) in runs.items():
    with training_progress(seed, TRAIN_STAGE):
        run_stages(exp, manifests, stages=[TRAIN_STAGE], condition_ids=CONDITIONS, device=DEVICE)


## 4. Grad-CAM

In [ ]:
import threading
import time
import inspect
from contextlib import contextmanager
from functools import wraps

@contextmanager
def activity_progress(seed, stage, interval=10):
    started = time.monotonic()
    status = {'phase': 'importing evaluation/reporting modules', 'condition': stage,
              'image': None, 'verified': set(), 'total': 0}
    patches = []
    stop = threading.Event()
    def emit(message):
        print(f'[seed={seed} {stage}] {message}', flush=True)
    def heartbeat():
        while not stop.wait(interval):
            message = f"{status['condition']} | {status['phase']} | elapsed {time.monotonic()-started:.0f}s"
            if status['total']:
                message += f" | verified {len(status['verified'])}/{status['total']} images"
            if status['image']:
                message += f" | current image {status['image']}"
            emit(message)
    def patch(module, name, replacement):
        patches.append((module, name, getattr(module, name)))
        setattr(module, name, replacement)
    emit('START: heartbeat every 10s')
    monitor = threading.Thread(target=heartbeat, daemon=True)
    monitor.start()
    try:
        import shared.xai as xai
        import shared.reporting as reporting
        from shared.preprocessing import ReleasePreprocessor
        def operation(module, name, label):
            original = getattr(module, name)
            @wraps(original)
            def wrapped(*args, **kwargs):
                status['phase'] = label
                emit(label)
                result = original(*args, **kwargs)
                status['phase'] = label + ' finished / continuing'
                return result
            patch(module, name, wrapped)
        operation(ReleasePreprocessor, 'prepare', 'checking/building preprocessing cache on VM')
        original_paths = xai._paths
        def paths(folder, image):
            status.update(image=image, phase='checking per-image artifacts / preparing explanation')
            return original_paths(folder, image)
        patch(xai, '_paths', paths)
        original_verify = xai.verify_record
        def verify(exp, criterion, condition_id, row, model_hash, folder=None):
            status.update(phase='verifying saved maps, overlays and localization metrics', image=row['image'])
            result = original_verify(exp, criterion, condition_id, row, model_hash, folder)
            status['verified'].add(row['image'])
            return result
        patch(xai, 'verify_record', verify)
        original_evaluate = xai.evaluate_condition
        defaults = inspect.signature(original_evaluate).parameters
        @wraps(original_evaluate)
        def evaluate(exp, manifests, criterion, condition_id, *args, **kwargs):
            status.update(condition=condition_id, phase='checking CNN checkpoint / loading model / preparing background',
                          image=None, verified=set(), total=len(manifests['xai']))
            emit(f"{criterion} {condition_id} START | {status['total']} images | checking completed CNN")
            for name, label in (('gradcam_fn', 'computing Grad-CAM'), ('shap_maps_fn', 'computing SHAP attributions')):
                fn = kwargs.get(name, defaults[name].default)
                def explain(*a, _fn=fn, _label=label, **k):
                    status['phase'] = _label
                    result = _fn(*a, **k)
                    status['phase'] = 'explanation calculated / preparing output artifacts'
                    return result
                kwargs[name] = explain
            result = original_evaluate(exp, manifests, criterion, condition_id, *args, **kwargs)
            status.update(phase='method finished', image=None)
            emit(f"{criterion} {condition_id} DONE | {len(status['verified'])}/{status['total']} verified images")
            return result
        patch(xai, 'evaluate_condition', evaluate)
        original_marker = reporting._read_marker
        def marker(exp, criterion, condition):
            status.update(condition=condition, phase=f'checking {criterion} completion and hashes', image=None, total=0)
            emit(f'{condition} | checking {criterion} results')
            return original_marker(exp, criterion, condition)
        patch(reporting, '_read_marker', marker)
        for name, label in (('classification_bootstrap', 'classification patient bootstrap CI / paired comparison'),
                            ('patient_bootstrap', 'XAI patient bootstrap CI / paired comparison'),
                            ('_strength_plots', 'generating strength comparison plots'),
                            ('summarize_xai', 'summarizing localization metrics and patient confidence intervals'),
                            ('report_all', 'collecting completed results / generating reports'),
                            ('summarize_seeds', 'verifying seed reports / calculating mean and SD')):
            operation(reporting, name, label)
        operation(xai, 'summarize_xai', 'summarizing localization metrics and patient confidence intervals')
        original_publish = xai.publish_file
        def publish(source, destination):
            status['phase'] = f'publishing and verifying Drive output: {destination}'
            result = original_publish(source, destination)
            status['phase'] = 'Drive output verified / continuing'
            return result
        patch(xai, 'publish_file', publish)
        status['phase'] = 'running selected stage'
        yield
    finally:
        stop.set()
        monitor.join(timeout=1)
        for module, name, original in reversed(patches):
            setattr(module, name, original)
        emit(f'cell ended after {time.monotonic()-started:.1f}s; hooks restored')

PIPELINE_STAGE = "gradcam"
for seed, (exp, manifests) in runs.items():
    with activity_progress(seed, PIPELINE_STAGE):
        run_stages(exp, manifests, stages=[PIPELINE_STAGE], condition_ids=CONDITIONS, device=DEVICE)


## 5. SHAP

In [ ]:
import threading
import time
import inspect
from contextlib import contextmanager
from functools import wraps

@contextmanager
def activity_progress(seed, stage, interval=10):
    started = time.monotonic()
    status = {'phase': 'importing evaluation/reporting modules', 'condition': stage,
              'image': None, 'verified': set(), 'total': 0}
    patches = []
    stop = threading.Event()
    def emit(message):
        print(f'[seed={seed} {stage}] {message}', flush=True)
    def heartbeat():
        while not stop.wait(interval):
            message = f"{status['condition']} | {status['phase']} | elapsed {time.monotonic()-started:.0f}s"
            if status['total']:
                message += f" | verified {len(status['verified'])}/{status['total']} images"
            if status['image']:
                message += f" | current image {status['image']}"
            emit(message)
    def patch(module, name, replacement):
        patches.append((module, name, getattr(module, name)))
        setattr(module, name, replacement)
    emit('START: heartbeat every 10s')
    monitor = threading.Thread(target=heartbeat, daemon=True)
    monitor.start()
    try:
        import shared.xai as xai
        import shared.reporting as reporting
        from shared.preprocessing import ReleasePreprocessor
        def operation(module, name, label):
            original = getattr(module, name)
            @wraps(original)
            def wrapped(*args, **kwargs):
                status['phase'] = label
                emit(label)
                result = original(*args, **kwargs)
                status['phase'] = label + ' finished / continuing'
                return result
            patch(module, name, wrapped)
        operation(ReleasePreprocessor, 'prepare', 'checking/building preprocessing cache on VM')
        original_paths = xai._paths
        def paths(folder, image):
            status.update(image=image, phase='checking per-image artifacts / preparing explanation')
            return original_paths(folder, image)
        patch(xai, '_paths', paths)
        original_verify = xai.verify_record
        def verify(exp, criterion, condition_id, row, model_hash, folder=None):
            status.update(phase='verifying saved maps, overlays and localization metrics', image=row['image'])
            result = original_verify(exp, criterion, condition_id, row, model_hash, folder)
            status['verified'].add(row['image'])
            return result
        patch(xai, 'verify_record', verify)
        original_evaluate = xai.evaluate_condition
        defaults = inspect.signature(original_evaluate).parameters
        @wraps(original_evaluate)
        def evaluate(exp, manifests, criterion, condition_id, *args, **kwargs):
            status.update(condition=condition_id, phase='checking CNN checkpoint / loading model / preparing background',
                          image=None, verified=set(), total=len(manifests['xai']))
            emit(f"{criterion} {condition_id} START | {status['total']} images | checking completed CNN")
            for name, label in (('gradcam_fn', 'computing Grad-CAM'), ('shap_maps_fn', 'computing SHAP attributions')):
                fn = kwargs.get(name, defaults[name].default)
                def explain(*a, _fn=fn, _label=label, **k):
                    status['phase'] = _label
                    result = _fn(*a, **k)
                    status['phase'] = 'explanation calculated / preparing output artifacts'
                    return result
                kwargs[name] = explain
            result = original_evaluate(exp, manifests, criterion, condition_id, *args, **kwargs)
            status.update(phase='method finished', image=None)
            emit(f"{criterion} {condition_id} DONE | {len(status['verified'])}/{status['total']} verified images")
            return result
        patch(xai, 'evaluate_condition', evaluate)
        original_marker = reporting._read_marker
        def marker(exp, criterion, condition):
            status.update(condition=condition, phase=f'checking {criterion} completion and hashes', image=None, total=0)
            emit(f'{condition} | checking {criterion} results')
            return original_marker(exp, criterion, condition)
        patch(reporting, '_read_marker', marker)
        for name, label in (('classification_bootstrap', 'classification patient bootstrap CI / paired comparison'),
                            ('patient_bootstrap', 'XAI patient bootstrap CI / paired comparison'),
                            ('_strength_plots', 'generating strength comparison plots'),
                            ('summarize_xai', 'summarizing localization metrics and patient confidence intervals'),
                            ('report_all', 'collecting completed results / generating reports'),
                            ('summarize_seeds', 'verifying seed reports / calculating mean and SD')):
            operation(reporting, name, label)
        operation(xai, 'summarize_xai', 'summarizing localization metrics and patient confidence intervals')
        original_publish = xai.publish_file
        def publish(source, destination):
            status['phase'] = f'publishing and verifying Drive output: {destination}'
            result = original_publish(source, destination)
            status['phase'] = 'Drive output verified / continuing'
            return result
        patch(xai, 'publish_file', publish)
        status['phase'] = 'running selected stage'
        yield
    finally:
        stop.set()
        monitor.join(timeout=1)
        for module, name, original in reversed(patches):
            setattr(module, name, original)
        emit(f'cell ended after {time.monotonic()-started:.1f}s; hooks restored')

PIPELINE_STAGE = "shap"
for seed, (exp, manifests) in runs.items():
    with activity_progress(seed, PIPELINE_STAGE):
        run_stages(exp, manifests, stages=[PIPELINE_STAGE], condition_ids=CONDITIONS, device=DEVICE)


## 6. ตาราง กราฟ CI และ paired comparisons

In [ ]:
import threading
import time
import inspect
from contextlib import contextmanager
from functools import wraps

@contextmanager
def activity_progress(seed, stage, interval=10):
    started = time.monotonic()
    status = {'phase': 'importing evaluation/reporting modules', 'condition': stage,
              'image': None, 'verified': set(), 'total': 0}
    patches = []
    stop = threading.Event()
    def emit(message):
        print(f'[seed={seed} {stage}] {message}', flush=True)
    def heartbeat():
        while not stop.wait(interval):
            message = f"{status['condition']} | {status['phase']} | elapsed {time.monotonic()-started:.0f}s"
            if status['total']:
                message += f" | verified {len(status['verified'])}/{status['total']} images"
            if status['image']:
                message += f" | current image {status['image']}"
            emit(message)
    def patch(module, name, replacement):
        patches.append((module, name, getattr(module, name)))
        setattr(module, name, replacement)
    emit('START: heartbeat every 10s')
    monitor = threading.Thread(target=heartbeat, daemon=True)
    monitor.start()
    try:
        import shared.xai as xai
        import shared.reporting as reporting
        from shared.preprocessing import ReleasePreprocessor
        def operation(module, name, label):
            original = getattr(module, name)
            @wraps(original)
            def wrapped(*args, **kwargs):
                status['phase'] = label
                emit(label)
                result = original(*args, **kwargs)
                status['phase'] = label + ' finished / continuing'
                return result
            patch(module, name, wrapped)
        operation(ReleasePreprocessor, 'prepare', 'checking/building preprocessing cache on VM')
        original_paths = xai._paths
        def paths(folder, image):
            status.update(image=image, phase='checking per-image artifacts / preparing explanation')
            return original_paths(folder, image)
        patch(xai, '_paths', paths)
        original_verify = xai.verify_record
        def verify(exp, criterion, condition_id, row, model_hash, folder=None):
            status.update(phase='verifying saved maps, overlays and localization metrics', image=row['image'])
            result = original_verify(exp, criterion, condition_id, row, model_hash, folder)
            status['verified'].add(row['image'])
            return result
        patch(xai, 'verify_record', verify)
        original_evaluate = xai.evaluate_condition
        defaults = inspect.signature(original_evaluate).parameters
        @wraps(original_evaluate)
        def evaluate(exp, manifests, criterion, condition_id, *args, **kwargs):
            status.update(condition=condition_id, phase='checking CNN checkpoint / loading model / preparing background',
                          image=None, verified=set(), total=len(manifests['xai']))
            emit(f"{criterion} {condition_id} START | {status['total']} images | checking completed CNN")
            for name, label in (('gradcam_fn', 'computing Grad-CAM'), ('shap_maps_fn', 'computing SHAP attributions')):
                fn = kwargs.get(name, defaults[name].default)
                def explain(*a, _fn=fn, _label=label, **k):
                    status['phase'] = _label
                    result = _fn(*a, **k)
                    status['phase'] = 'explanation calculated / preparing output artifacts'
                    return result
                kwargs[name] = explain
            result = original_evaluate(exp, manifests, criterion, condition_id, *args, **kwargs)
            status.update(phase='method finished', image=None)
            emit(f"{criterion} {condition_id} DONE | {len(status['verified'])}/{status['total']} verified images")
            return result
        patch(xai, 'evaluate_condition', evaluate)
        original_marker = reporting._read_marker
        def marker(exp, criterion, condition):
            status.update(condition=condition, phase=f'checking {criterion} completion and hashes', image=None, total=0)
            emit(f'{condition} | checking {criterion} results')
            return original_marker(exp, criterion, condition)
        patch(reporting, '_read_marker', marker)
        for name, label in (('classification_bootstrap', 'classification patient bootstrap CI / paired comparison'),
                            ('patient_bootstrap', 'XAI patient bootstrap CI / paired comparison'),
                            ('_strength_plots', 'generating strength comparison plots'),
                            ('summarize_xai', 'summarizing localization metrics and patient confidence intervals'),
                            ('report_all', 'collecting completed results / generating reports'),
                            ('summarize_seeds', 'verifying seed reports / calculating mean and SD')):
            operation(reporting, name, label)
        operation(xai, 'summarize_xai', 'summarizing localization metrics and patient confidence intervals')
        original_publish = xai.publish_file
        def publish(source, destination):
            status['phase'] = f'publishing and verifying Drive output: {destination}'
            result = original_publish(source, destination)
            status['phase'] = 'Drive output verified / continuing'
            return result
        patch(xai, 'publish_file', publish)
        status['phase'] = 'running selected stage'
        yield
    finally:
        stop.set()
        monitor.join(timeout=1)
        for module, name, original in reversed(patches):
            setattr(module, name, original)
        emit(f'cell ended after {time.monotonic()-started:.1f}s; hooks restored')

PIPELINE_STAGE = "reports"
for seed, (exp, manifests) in runs.items():
    with activity_progress(seed, PIPELINE_STAGE):
        run_stages(exp, manifests, stages=[PIPELINE_STAGE], condition_ids=CONDITIONS, device=DEVICE)


## 7. รวมคะแนนแต่ละ seed (mean±SD แยกจาก patient CI)

In [ ]:
import threading
import time
import inspect
from contextlib import contextmanager
from functools import wraps

@contextmanager
def activity_progress(seed, stage, interval=10):
    started = time.monotonic()
    status = {'phase': 'importing evaluation/reporting modules', 'condition': stage,
              'image': None, 'verified': set(), 'total': 0}
    patches = []
    stop = threading.Event()
    def emit(message):
        print(f'[seed={seed} {stage}] {message}', flush=True)
    def heartbeat():
        while not stop.wait(interval):
            message = f"{status['condition']} | {status['phase']} | elapsed {time.monotonic()-started:.0f}s"
            if status['total']:
                message += f" | verified {len(status['verified'])}/{status['total']} images"
            if status['image']:
                message += f" | current image {status['image']}"
            emit(message)
    def patch(module, name, replacement):
        patches.append((module, name, getattr(module, name)))
        setattr(module, name, replacement)
    emit('START: heartbeat every 10s')
    monitor = threading.Thread(target=heartbeat, daemon=True)
    monitor.start()
    try:
        import shared.xai as xai
        import shared.reporting as reporting
        from shared.preprocessing import ReleasePreprocessor
        def operation(module, name, label):
            original = getattr(module, name)
            @wraps(original)
            def wrapped(*args, **kwargs):
                status['phase'] = label
                emit(label)
                result = original(*args, **kwargs)
                status['phase'] = label + ' finished / continuing'
                return result
            patch(module, name, wrapped)
        operation(ReleasePreprocessor, 'prepare', 'checking/building preprocessing cache on VM')
        original_paths = xai._paths
        def paths(folder, image):
            status.update(image=image, phase='checking per-image artifacts / preparing explanation')
            return original_paths(folder, image)
        patch(xai, '_paths', paths)
        original_verify = xai.verify_record
        def verify(exp, criterion, condition_id, row, model_hash, folder=None):
            status.update(phase='verifying saved maps, overlays and localization metrics', image=row['image'])
            result = original_verify(exp, criterion, condition_id, row, model_hash, folder)
            status['verified'].add(row['image'])
            return result
        patch(xai, 'verify_record', verify)
        original_evaluate = xai.evaluate_condition
        defaults = inspect.signature(original_evaluate).parameters
        @wraps(original_evaluate)
        def evaluate(exp, manifests, criterion, condition_id, *args, **kwargs):
            status.update(condition=condition_id, phase='checking CNN checkpoint / loading model / preparing background',
                          image=None, verified=set(), total=len(manifests['xai']))
            emit(f"{criterion} {condition_id} START | {status['total']} images | checking completed CNN")
            for name, label in (('gradcam_fn', 'computing Grad-CAM'), ('shap_maps_fn', 'computing SHAP attributions')):
                fn = kwargs.get(name, defaults[name].default)
                def explain(*a, _fn=fn, _label=label, **k):
                    status['phase'] = _label
                    result = _fn(*a, **k)
                    status['phase'] = 'explanation calculated / preparing output artifacts'
                    return result
                kwargs[name] = explain
            result = original_evaluate(exp, manifests, criterion, condition_id, *args, **kwargs)
            status.update(phase='method finished', image=None)
            emit(f"{criterion} {condition_id} DONE | {len(status['verified'])}/{status['total']} verified images")
            return result
        patch(xai, 'evaluate_condition', evaluate)
        original_marker = reporting._read_marker
        def marker(exp, criterion, condition):
            status.update(condition=condition, phase=f'checking {criterion} completion and hashes', image=None, total=0)
            emit(f'{condition} | checking {criterion} results')
            return original_marker(exp, criterion, condition)
        patch(reporting, '_read_marker', marker)
        for name, label in (('classification_bootstrap', 'classification patient bootstrap CI / paired comparison'),
                            ('patient_bootstrap', 'XAI patient bootstrap CI / paired comparison'),
                            ('_strength_plots', 'generating strength comparison plots'),
                            ('summarize_xai', 'summarizing localization metrics and patient confidence intervals'),
                            ('report_all', 'collecting completed results / generating reports'),
                            ('summarize_seeds', 'verifying seed reports / calculating mean and SD')):
            operation(reporting, name, label)
        operation(xai, 'summarize_xai', 'summarizing localization metrics and patient confidence intervals')
        original_publish = xai.publish_file
        def publish(source, destination):
            status['phase'] = f'publishing and verifying Drive output: {destination}'
            result = original_publish(source, destination)
            status['phase'] = 'Drive output verified / continuing'
            return result
        patch(xai, 'publish_file', publish)
        status['phase'] = 'running selected stage'
        yield
    finally:
        stop.set()
        monitor.join(timeout=1)
        for module, name, original in reversed(patches):
            setattr(module, name, original)
        emit(f'cell ended after {time.monotonic()-started:.1f}s; hooks restored')

with activity_progress("all", "seed_summary"):
    from shared.reporting import summarize_seeds
    summarize_seeds(next(iter(runs.values()))[0].base_output_root)
